# 🎯 Notebook 08 — Model Recommendation & Inference Tool
**NB6057CEM CW1 | T5 (8 marks)**

Builds the final comparison table, selects and justifies the recommended model, and provides a runnable inference pipeline.

---

In [ ]:
import torch, torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json, os, time
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('splits/preprocessing.json') as f:
    prep = json.load(f)
MEAN    = np.array(prep['mean'], dtype=np.float32)
STD     = np.array(prep['std'], dtype=np.float32)
CLASSES = prep['classes']
print(f'Classes: {CLASSES}')

## T5a Model Recommendation Table

Compare all 3 models on: mean macro-F1, std, parameters, and inference time.

In [ ]:
# Load saved results
baseline = json.load(open('metrics/baseline_result.json'))
mlp_res  = json.load(open('metrics/mlp_results.json'))
cnn_res  = json.load(open('metrics/cnn_results.json'))

print(f"{'Model':<20} {'Test F1 mean':>12} {'Test F1 std':>12} {'Params':>10} {'Infer ms':>10}")
print('-' * 70)
print(f"{'Baseline (HOG+LR)':<20} {baseline['test_macro_f1']:>12.4f} {'N/A':>12} {baseline['params']:>10,} {baseline['inference_ms']:>10.1f}")
print(f"{'MLP (PyTorch)':<20} {mlp_res['test_mean']:>12.4f} {mlp_res['test_std']:>12.4f} {'~6,425,610':>10} {'TBD':>10}")
print(f"{'CNN (PyTorch)':<20} {cnn_res['test_mean']:>12.4f} {cnn_res['test_std']:>12.4f} {'~94,730':>10} {'TBD':>10}")

print()
print('Recommendation: CNN is recommended if GPU is available — it achieves highest F1')
print('with far fewer parameters (~94K vs ~6.4M for MLP), showing better generalisation.')
print('If only CPU with no GPU and speed matters, Baseline may be acceptable if its F1')
print('is within a small margin and no retraining is needed.')

## Measure Inference Time (batch=64, Colab CPU/GPU)

In [ ]:
def conv_block(i, o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
def make_cnn(p=0.3):
    return nn.Sequential(conv_block(3,32), conv_block(32,64), conv_block(64,128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p), nn.Linear(128,10))
def make_mlp(p=0.3):
    return nn.Sequential(nn.Flatten(), nn.Linear(64*64*3,512), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(512,256), nn.ReLU(), nn.Dropout(p), nn.Linear(256,10))

best_cnn_seed = max(cnn_res['seeds'], key=lambda r: r['val_macro_f1'])['seed']
best_mlp_seed = max(mlp_res['seeds'], key=lambda r: r['val_macro_f1'])['seed']

cnn = make_cnn().to(DEVICE); cnn.load_state_dict(torch.load(f'checkpoints/cnn_seed{best_cnn_seed}.pt')); cnn.eval()
mlp = make_mlp().to(DEVICE); mlp.load_state_dict(torch.load(f'checkpoints/mlp_seed{best_mlp_seed}.pt')); mlp.eval()

dummy = torch.randn(64, 3, 64, 64).to(DEVICE)

def measure_ms(model, batch, n_runs=20, warmup=5):
    with torch.no_grad():
        for _ in range(warmup): model(batch)
        t0 = time.time()
        for _ in range(n_runs): model(batch)
        return (time.time() - t0) / n_runs * 1000

mlp_ms = measure_ms(mlp, dummy)
cnn_ms = measure_ms(cnn, dummy)

print(f'MLP inference: {mlp_ms:.2f} ms (batch=64, {DEVICE})')
print(f'CNN inference: {cnn_ms:.2f} ms (batch=64, {DEVICE})')

## T5b Inference Pipeline (`src/infer.py`)

The same function also lives in `src/infer.py` for command-line use.

In [ ]:
def predict(image_path, model, mean, std, classes, threshold=0.6):
    """Predict land-use class for a single image file.

    Args:
        image_path : str — path to image file
        model      : PyTorch model in eval mode
        mean, std  : array-like — per-channel normalisation stats (from train split)
        classes    : list[str] — class names
        threshold  : float — confidence threshold for OOD warning

    Returns:
        dict with 'prediction' or 'warning' or 'error'
    """
    try:
        img = Image.open(image_path).convert('RGB').resize((64, 64))
    except Exception as e:
        return {'error': f'Cannot read image: {e}'}

    x = (np.asarray(img, dtype=np.float32) / 255.0 - mean) / std
    x = torch.tensor(x).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]

    conf, k = probs.max(0)
    conf = conf.item()

    if conf < threshold:
        return {
            'warning': 'Low confidence — image may be out of scope.',
            'best_guess': classes[k],
            'confidence': round(conf, 3),
            'note': 'Softmax confidence is imperfect for OOD detection; networks can be confidently wrong on unfamiliar inputs.'
        }
    return {'prediction': classes[k], 'confidence': round(conf, 3)}

print('predict() function defined.')

## Demo — Normal satellite image

In [ ]:
data   = np.load('/content/drive/MyDrive/ann_cw1/eurosat_64.npz')
X_full = data['X']
splits = np.load('splits/split_indices.npz')
test_idx = splits['test']

sample_img = X_full[test_idx[0]]
sample_path = '/tmp/sample_test_image.jpg'
Image.fromarray(sample_img).save(sample_path)

result = predict(sample_path, cnn, MEAN, STD, CLASSES, threshold=0.6)
print('Normal satellite image:')
print(result)

fig, ax = plt.subplots(figsize=(3,3))
ax.imshow(sample_img); ax.axis('off')
ax.set_title(f"Pred: {result.get('prediction', result.get('best_guess', 'N/A'))}\n"
             f"Conf: {result.get('confidence', 'N/A')}", fontsize=9)
plt.tight_layout()
plt.savefig('figures/inference_demo_valid.png', dpi=150)
plt.show()

## Demo — Invalid / out-of-scope input

In [ ]:
with open('/tmp/not_an_image.txt', 'w') as f:
    f.write('This is not an image.')
result_bad = predict('/tmp/not_an_image.txt', cnn, MEAN, STD, CLASSES)
print('Invalid file (text):', result_bad)

noise_img = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
noise_path = '/tmp/random_noise.jpg'
Image.fromarray(noise_img).save(noise_path)
result_noise = predict(noise_path, cnn, MEAN, STD, CLASSES, threshold=0.6)
print('Out-of-scope (noise image):', result_noise)

## T5c Limitations Summary

1. **Dataset bias:** EuroSAT covers only European regions, one satellite (Sentinel-2), one season range, and cloud-free patches only.
2. **Generalisation:** Untested on other regions, sensors, or resolutions. Random split may be optimistic due to spatial correlation of neighbouring patches.
3. **Misuse and privacy:** Land-use classification could support unwanted surveillance or automated planning decisions without human oversight.
4. **Feasible next experiment:** Test on a geographically diverse dataset (e.g., BigEarthNet) or add the 13-band multispectral EuroSAT to explore whether infrared bands improve classification.
5. **OOD detection limitation:** Softmax confidence thresholding is imperfect — neural networks can be confidently wrong on inputs outside their training distribution. Do not claim deployment readiness from a single benchmark.

---
## ✅ Notebook 08 Complete
**Final steps:** Write `README.md`, run smoke test, record demo, write technical record (≤1500 words), and prepare ZIP for submission.